Импорты и данные

In [32]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from scipy.stats import randint, uniform

In [34]:
df = pd.read_csv(r"C:\Users\aleks\Downloads\_train_sem09__1_\_train_sem09 (1).csv")  
X = df.drop(columns=['Activity'])
y = df['Activity']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

Базовые модели

In [30]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

pred_lr = lr.predict(X_test)
f1_lr = f1_score(y_test, pred_lr)

print("Baseline LR:", round(f1_lr, 2))

Baseline LR: 0.78


In [20]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)
f1_rf = f1_score(y_test, pred_rf)
print("Baseline RandomForest F1:", round(f1_rf, 2))

Baseline RandomForest F1: 0.8


GridSearchCV

In [21]:
grid_lr = {
    'C': [0.1, 1, 10],
    'penalty': ['l2'],
    'solver': ['lbfgs']
}

gs_lr = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    grid_lr,
    scoring='f1',
    cv=cv,
    n_jobs=-1
)
gs_lr.fit(X_train, y_train)

pred_gs_lr = gs_lr.best_estimator_.predict(X_test)
print("GridSearchCV LogisticRegression F1:", round(f1_score(y_test, pred_gs_lr), 2))
print(gs_lr.best_params_)

GridSearchCV LogisticRegression F1: 0.79
{'C': 0.1, 'penalty': 'l2', 'solver': 'lbfgs'}


RandomizedSearchCV

In [22]:
rand_rf = {
    'n_estimators': randint(50, 201),
    'max_depth': randint(2, 21),
    'min_samples_leaf': randint(1, 11)
}

rs_rf = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    rand_rf,
    n_iter=20,
    scoring='f1',
    cv=cv,
    random_state=42,
    n_jobs=-1
)
rs_rf.fit(X_train, y_train)

pred_rs_rf = rs_rf.best_estimator_.predict(X_test)
print("RandomizedSearchCV RandomForest F1:", round(f1_score(y_test, pred_rs_rf), 2))
print(rs_rf.best_params_)

RandomizedSearchCV RandomForest F1: 0.8
{'max_depth': 16, 'min_samples_leaf': 3, 'n_estimators': 130}


Hyperopt

In [27]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
import warnings
warnings.filterwarnings("ignore")

space = {
    'C': hp.loguniform('C', np.log(0.01), np.log(10)),
    'solver': hp.choice('solver', ['lbfgs']),
    'penalty': hp.choice('penalty', ['l2'])
}

def objective(params):
    model = LogisticRegression(
        max_iter=1000,
        random_state=42,
        **params
    )
    scores = []
    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        model.fit(X_tr, y_tr)
        pred = model.predict(X_val)
        scores.append(f1_score(y_val, pred))
    return {'loss': -np.mean(scores), 'status': STATUS_OK}

trials = Trials()
best_hyperopt = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=20,
    trials=trials,
    rstate=np.random.default_rng(42)
)

print("Hyperopt done")

  0%|          | 0/20 [00:00<?, ?trial/s, best loss=?]

100%|██████████| 20/20 [01:34<00:00,  4.71s/trial, best loss: -0.7902797917514244]
Hyperopt done


Optuna

In [28]:
import optuna

def objective_optuna(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 2, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

    model = RandomForestClassifier(
        random_state=42,
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf
    )

    scores = []
    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        model.fit(X_tr, y_tr)
        pred = model.predict(X_val)
        scores.append(f1_score(y_val, pred))
    return np.mean(scores)

study = optuna.create_study(direction='maximize')
study.optimize(objective_optuna, n_trials=20)

print("Optuna best value:", study.best_value)
print("Optuna best params:", study.best_params)

[I 2026-08-05 16:46:29,219] A new study created in memory with name: no-name-aa31dd60-4651-4429-b2a0-3d7a6d60f579


[I 2026-08-05 16:46:34,589] Trial 0 finished with value: 0.7487705397706663 and parameters: {'n_estimators': 162, 'max_depth': 3, 'min_samples_leaf': 3}. Best is trial 0 with value: 0.7487705397706663.
[I 2026-08-05 16:46:40,916] Trial 1 finished with value: 0.7957565893614121 and parameters: {'n_estimators': 88, 'max_depth': 20, 'min_samples_leaf': 9}. Best is trial 1 with value: 0.7957565893614121.
[I 2026-08-05 16:46:54,607] Trial 2 finished with value: 0.8067299932609597 and parameters: {'n_estimators': 194, 'max_depth': 11, 'min_samples_leaf': 5}. Best is trial 2 with value: 0.8067299932609597.
[I 2026-08-05 16:46:59,842] Trial 3 finished with value: 0.8047674055187981 and parameters: {'n_estimators': 66, 'max_depth': 13, 'min_samples_leaf': 4}. Best is trial 2 with value: 0.8067299932609597.
[I 2026-08-05 16:47:10,610] Trial 4 finished with value: 0.7971386774636199 and parameters: {'n_estimators': 173, 'max_depth': 10, 'min_samples_leaf': 7}. Best is trial 2 with value: 0.806729

Optuna best value: 0.816985330007147
Optuna best params: {'n_estimators': 123, 'max_depth': 19, 'min_samples_leaf': 2}


Финальное сравнение

In [35]:
print("Baseline LR:", round(f1_lr, 2))
print("Baseline RF:", round(f1_rf, 2))
print("GridSearchCV LR:", round(f1_score(y_test, pred_gs_lr), 2))
print("RandomizedSearchCV RF:", round(f1_score(y_test, pred_rs_rf), 2))
print("Hyperopt best:", round(np.max([-t['result']['loss'] for t in trials.trials]), 2))
print("Optuna best CV:", round(study.best_value, 2))

Baseline LR: 0.78
Baseline RF: 0.8
GridSearchCV LR: 0.79
RandomizedSearchCV RF: 0.8
Hyperopt best: 0.79
Optuna best CV: 0.82
